# 💻 Notebook do Aluno — Aula 11: 🎯 Aula Integradora Agente + RAG como tool + Gradio ao vivo

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 11/14 — Lab 100% · Sem conceito novo · CKP03 entrega**  
**⏱️ 1h40min · 100% Lab**  
**🏁 CKP03 · create_retriever_tool · ngrok**  
**🔁 Andaime 60%**  

---

## 🎯 Objetivo da aula

Aula 11 (hoje): create_retriever_tool() + agente 3 tools + Gradio + ngrok = URL pública ao vivo · CKP03 entregue

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 11 · CKP03 · Entrega obrigatória**  
### Agente Inteligente com RAG como tool + Gradio ao vivo ★★★

*Grupo 3–4 · 25 minutos em aula + complementar fora*

1. Complete as 6 lacunas do notebook: persist_directory, k=3, description da tool_rag (mais importante!), lista de tools, agente+executor, chat_stream com streaming, Gradio com 3 exemplos do domínio e share=True.
2. Valide as 4 ferramentas antes de subir para o Gradio: testar tool_rag.invoke(), buscar_na_web.invoke(), calcular.invoke() e uma pergunta multi-step (RAG + calc).
3. Publique a URL com share=True. Copie a URL e envie no canal da turma.
4. Demo cruzada : acessar a URL do grupo vizinho e fazer 2 perguntas. Registrar: o agente respondeu corretamente? Usou a tool certa?
5. Documentar 3 exemplos de interação no notebook (célula markdown) — pergunta, tool usada, qualidade da resposta.

> **🎯 Gabarito das lacunas**
>
> L1: persist_directory="/content/ckp02", k=3
>
> L2: description descrevendo o domínio específico + quando NÃO usar
>
> L3: description com "informações atuais... NÃO use para conteúdo interno"
>
> L4: tools como 1º arg, tools=tools, max_iterations=5
>
> L5: yield "⚠️ Bloqueado..." ; yield resp (streaming acumulado)
>
> L6: fn=chat_stream, title="🤖 [Domínio] CKP03", examples=[...], share=True

---

## 🧩 Notebook Aluno — setup e tools (60% lacunas)

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-community langchain-chroma duckduckgo-search gradio tiktoken -q

import os
from google.colab import userdata
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.tools.retriever import create_retriever_tool
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
import gradio as gr

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 👉 LACUNA 1: carregar o ChromaDB do CKP02 e criar o retriever
db        = Chroma(persist_directory=___, embedding_function=embeddings)
retriever = db.as_retriever(search_kwargs={"k": ___})  # k=3 recomendado

# 👉 LACUNA 2: criar a tool RAG com create_retriever_tool()
# Substituir [DOMINIO] pelo domínio real do grupo
tool_rag = create_retriever_tool(
    retriever,
    name="buscar_no_dominio",
    description="""___""",  # quando usar, quando NÃO usar, o que retorna
)

# Chain de compressão (copiada da Aula 10)
chain_resumir = (
    ChatPromptTemplate.from_template("Resuma em 3 frases:\n\n{texto}")
    | llm | StrOutputParser()
)

# 👉 LACUNA 3: implementar buscar_na_web com compressão
@tool
def buscar_na_web(query: str) -> str:
    """___"""  # escrever description (quando usar, quando NÃO usar)
    bruto = DuckDuckGoSearchRun().run(query)
    return chain_resumir.invoke({"texto": bruto}) if len(bruto) > 400 else bruto

@tool
def calcular(expressao: str) -> str:
    """Use para cálculos matemáticos com expressão Python. NÃO use para busca."""
    try: return str(eval(expressao,{"__builtins__":{}},{}))
    except Exception as e: return f"Erro: {e}"

tools = [tool_rag, buscar_na_web, calcular]

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Docs LangChain — create_retriever_tool: documentação oficial do helper para encapsular retrievers como tools de agente. python.langchain.com/docs/how_to/qa_sources
- Docs Gradio — ChatInterface com streaming e deploy. gradio.app/docs/gradio/chatinterface
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Fundamentação do princípio da ação mínima aplicado nesta integração. anthropic.com/engineering/building-effective-agents
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes racionais: o modelo percepção-ação-ambiente que fundamenta todo o Módulo 3 desta disciplina.
- Livro Polzer, D. — RAG with Python Cookbook. O'Reilly, 2026. Tool design de responsabilidade única e tratamento de erro para dependências externas — os dois princípios por trás da tool_rag desta integração.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 13 — Human-in-the-Loop: papéis do humano e Escalation Policies por trás do guardrail de tópicos proibidos desta aula.
- Paper Yao, S. et al. — "ReAct: Synergizing Reasoning and Acting in Language Models." ICLR, 2023. O padrão ReAct implementado nas Aulas 09–11. arxiv.org/abs/2210.03629

---

**Próxima Aula — Aula 12** — Router chains e o conceito de grafo de estado
  
Router Chain classifica intenção e roteia para handlers — e o grafo de estado entra como modelo mental, comparando AgentExecutor vs. StateGraph antes do código.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*